In [ ]:
!pip install langchain langchain-community chromadb tiktoken pypdf sentence-transformers langchain-openai -q

In [ ]:
!pip install faiss-cpu -q
print("✅ FAISS install ho gaya!")

In [ ]:
from google.colab import files
import os

os.makedirs("docs", exist_ok=True)

uploaded = files.upload()  # File picker khulega

for filename in uploaded.keys():
    os.rename(filename, f"docs/{filename}")
    print(f"✅ File save ho gayi: {filename}")

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import os

pdf_file = [f for f in os.listdir("docs") if f.endswith(".pdf")][0]
loader = PyPDFLoader(f"docs/{pdf_file}")
documents = loader.load()

# ✅ Chunks bade kar diye
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,   # 500 se 1000 kar diya
    chunk_overlap=150  # 50 se 150 kar diya
)
chunks = splitter.split_documents(documents)
print(f"✅ {len(chunks)} chunks ban gaye")

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2"
)

vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)

print("✅ FAISS Vector store ready!")

In [ ]:
!pip install langchain-groq -q
print("✅ Done!")

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
import os

# ⚠️ Yahan apni GROQ key paste karo - gsk_ se shuru hogi
# Apni notebook mein key ki jagah yeh likho
GROQ_KEY = "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"  # ← key hata do  # ← apni real key yahan likho

# Retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# Prompt
prompt_template = PromptTemplate(
    template="""You are a helpful assistant. Use ONLY the context below to answer.
If the answer is not in the context, say "Mujhe is baare mein information nahi hai."

Context:
{context}

Question: {question}

Answer:""",
    input_variables=["context", "question"]
)

# Groq LLM
llm = ChatGroq(
    model="llama3-8b-8192",
    temperature=0,
    api_key=GROQ_KEY
)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Naya chain
qa_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt_template
    | llm
    | StrOutputParser()
)

print("✅ Groq Chatbot ready hai!")

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# ✅ Smarter retriever
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 6,
        "fetch_k": 12,
        "lambda_mult": 0.7
    }
)

# ✅ Behtar prompt
prompt_template = PromptTemplate(
    template="""You are an expert assistant specifically about Pakistan Air Force.
Your job is to give detailed, accurate, and well-structured answers.

Instructions:
- Answer in detail using ALL relevant information from context
- If dates or numbers are mentioned, include them
- Structure your answer in points if multiple facts exist
- If context has partial info, share what you know
- Only say "information nahi hai" if context has absolutely nothing

Context:
{context}

Question: {question}

Detailed Answer:""",
    input_variables=["context", "question"]
)

# ✅ LLM
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.2,
    max_tokens=1024,
    api_key=GROQ_KEY
)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

qa_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt_template
    | llm
    | StrOutputParser()
)

print("✅ Improved chatbot ready!")

In [ ]:
import nbformat

# Notebook file path
notebook_path = "pafchatbot.ipynb"

# Colab se download karo pehle - File -> Download -> .ipynb
# Phir yeh Colab cell chalao:

!jupyter nbconvert --to notebook --ClearMetadataPreprocessor.enabled=True \
  --ClearMetadataPreprocessor.preserve_nb_metadata=True \
  pafchatbot.ipynb --output pafchatbot_clean.ipynb

print("✅ Clean notebook ready!")

In [ ]:
!ls /content/

In [ ]:
from ipywidgets import widgets
from IPython.display import display, clear_output

text_input = widgets.Text(
    placeholder="Learn about PAF",
    layout=widgets.Layout(width="70%")
)
button = widgets.Button(
    description="ASK",
    button_style="primary"
)
output = widgets.Output()

def on_ask(b):
    with output:
        clear_output()
        question = text_input.value.strip()
        if question:
            print(f"🧑 YOU: {question}\n")
            print("⏳ THINKING...\n")
            docs = retriever.invoke(question)
            result = qa_chain.invoke(question)
            print(f"🤖 Bot:\n{result}")
            print("\n" + "─"*50)
            print("📄 Sources:")
            for i, doc in enumerate(docs[:3], 1):
                print(f"\n[{i}] Page {doc.metadata.get('page', '?') + 1}:")
                print(f"{doc.page_content[:150]}...")

button.on_click(on_ask)
display(widgets.HBox([text_input, button]), output)